# 01. EDA y Limpieza de Datos

Carga, exploración y limpieza inicial del dataset CFPB.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_RAW = Path("data/raw/muestra_nlp_limpia.csv")
DATA_INTERIM = Path("data/interim")
DATA_INTERIM.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
# Carga con manejo de línea corrupta
df = pd.read_csv(DATA_RAW, engine="python", on_bad_lines="skip")
print(f"Filas: {len(df):,}")
print("Nulos:")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# EDA básico
narr = df["Consumer complaint narrative"].astype(str)
print("Longitud de narrativas:")
print(narr.str.len().describe())
print("\nCon XXXX:", narr.str.contains("XXXX", case=False).sum(), f"({narr.str.contains('XXXX', case=False).sum()/len(narr)*100:.1f}%)")


In [ ]:
# Limpieza
min_length = 20
short_mask = narr.str.len() < min_length
print(f"Eliminando {short_mask.sum()} narrativas cortas")
df_clean = df[~short_mask].copy()
df_clean["narrative_length"] = df_clean["Consumer complaint narrative"].astype(str).str.len()
df_clean["Date received"] = pd.to_datetime(df_clean["Date received"], errors="coerce")
df_clean["year"] = df_clean["Date received"].dt.year

df_clean.to_csv(DATA_INTERIM / "01_limpio.csv", index=False)
print(f"Guardado: {len(df_clean):,} filas")
